# Init

In [ ]:
import sys
from typing import Literal

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import skimage

sys.path.append('..')
from tests import test_image_warping

In [ ]:
plt.rcParams['figure.constrained_layout.use'] = True
np.set_printoptions(threshold=20, edgeitems=10, linewidth=140, precision=3, suppress=True)

# Task 1: implement linear interpolation in the `warp_inv` function

- Complete implementation of the `warp_inv` from the [geometric_transformations lecture](../lectures/geometric_transformations.ipynb) so that it supports linear interpolation by passing `interp='linear'`.
- It is *not* allowed to use `np.interp`, `scipy.interpolate`, etc.

<figure class="image">
  <img src="../figures/image_warping-expected_warp_inv_outputs.png" alt="" style="width: 12.8in;"/>
  <figcaption>Figure 1: Expected outputs. Notice that image on the right is not pixelated.</figcaption>
</figure>

In [ ]:
# Copy-pasted from the geometric_transformations lecture notebook
def compute_tform_span(img: np.ndarray, A: np.ndarray) -> tuple[tuple[int, int], tuple[float, float]]:
    h, w = img.shape[:2]

    # Get image corners and transform them
    X = np.array([0., w, w, 0.])
    Y = np.array([0., 0., h, h])
    XY = np.vstack((X, Y, np.ones(4)))
    XY_ = np.dot(A, XY)
    XY_ = XY_[:2] / XY_[2]  # normalize homogeneous coordinates - this is extra compared to affine case
    dx_, dy_ =  XY_[0, :].min(),  XY_[1, :].min()
    
    # Calculate the new size
    w_ = int(XY_[0, :].max() - XY_[0, :].min())
    h_ = int(XY_[1, :].max() - XY_[1, :].min())
    
    return (h_, w_), (dx_, dy_)

In [ ]:
# Copy-pasted from the geometric_transformations lecture notebook
def warp_inv(
    img: np.ndarray,
    A: np.ndarray,
    out_shape: tuple[int, int] | None = None,
    offset: tuple[float, float] = (0.0, 0.0),
    interp: Literal['nearest', 'linear'] = 'nearest',
) -> np.ndarray:
    # Input shape
    h, w = img.shape[:2]

    # Output shape
    if out_shape is None:
        h_, w_ = h, w
    else:
        h_, w_ = out_shape
    if img.ndim == 2:
        img_ = np.zeros((h_, w_), dtype=img.dtype).squeeze()
    else:
        img_ = np.zeros((h_, w_, img.shape[2]), dtype=img.dtype).squeeze()

    # Invert transformation
    A_inv = np.linalg.inv(A)

    # Helper function to safely get a value gray[y, x]
    def src_img_at(x, y):
        if 0 <= x < w and 0 <= y < h:
            return img[y, x]
        else:
            return 0

    # The main transformation loop
    dx_, dy_ = offset
    for y_ in range(h_):
        for x_ in range(w_):
            x, y, z = np.dot(A_inv, [x_ + dx_, y_ + dy_, 1.0])  # inverse transform
            x, y = x / z, y / z  # normalize homogeneous coordinates - this is extra compared to affine case
            if interp == 'nearest':
                x = int(0.5 + x)  # round
                y = int(0.5 + y)  # round
                img_[y_, x_] = src_img_at(x, y)  # src_img_at defined on line 16 of this function
            elif interp == 'linear':
                ########################################
                # TODO: implement
                
                raise NotImplementedError
            
                ########################################
            else:
                raise ValueError(interp)

    return img_

In [ ]:
rgb = skimage.io.imread('../data/robot.png')
rgb = skimage.transform.rescale(rgb, 0.5, channel_axis=2)
gray = skimage.color.rgb2gray(rgb)

A_hom = np.array([
    [ 1.8, -0.6, 100.],
    [-1.0,  2.0, 60.],
    [-0.004, -0.002, 1.0],
])

gray_hom_nn = warp_inv(gray, A_hom, *compute_tform_span(gray, A_hom), interp='nearest')
gray_hom_lin = warp_inv(gray, A_hom, *compute_tform_span(gray, A_hom), interp='linear')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
axes[0].imshow(gray, cmap='gray', vmin=0.0, vmax=1.0, interpolation='none')
axes[1].imshow(gray_hom_nn, cmap='gray', vmin=0.0, vmax=1.0, interpolation='none')
axes[2].imshow(gray_hom_lin, cmap='gray', vmin=0.0, vmax=1.0, interpolation='none');

In [ ]:
test_image_warping.TestWarpInv.eval(warp_inv_fn=warp_inv)

# Task 2: rotation around center of the image instead of origin=(0, 0)

- Implement the function `compute_rotation_matrix` that will compute a rotation matrix that if applied to an image, rotates around its center insteaf of the origin, see the figure below.
- It must work for arbitray angle `degrees`.
- Use the `warp_inv` function to demonstrate functionality for various angles.
- *Hint*: You need to set the translation offsets $t_x$ and $t_y$ to correct values.
- Example code:  
  ``` python
  rgb = skimage.io.imread('../data/robot.png')
  rot_mat = compute_rotation_matrix(angle, *rgb.shape[:2])
  rgb_rot = warp_inv(rgb, rot_mat, *calc_out_shape(rgb, rot_mat))
  ```
- Expected output:  
  <figure class="image">
    <img src="../figures/image_warping-expected_rot_outputs.png" alt="" style="width: 12.8in;"/>
    <figcaption>Figure 2: Expected outputs. Notice that images are rotated around their center.</figcaption>
  </figure>

In [ ]:
def compute_rotation_matrix(degrees: float, im_h: int, im_w: int) -> np.ndarray:
    """
    Compute the rotation matrix for a rotation around the center transformation.

    Args:
        degrees: The rotation angle in degrees.
        im_h: The height of the image.
        im_w: The width of the image.
    Returns:
        rot_mat: 3x3 rotation matrix as a numpy array suitable for the `warp_inv` function.
    """
    ########################################
    # TODO: implement
    
    raise NotImplementedError
    
    # ENDTODO
    ########################################

    return rot_mat


In [ ]:
rgb = skimage.io.imread('../data/robot.png')

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for deg, ax in zip([30, 60, -30], axes):
    # Prepare the rotation matrix
    rot_mat = compute_rotation_matrix(deg, *rgb.shape[:2])
    rgb_rot = warp_inv(rgb, rot_mat, *compute_tform_span(rgb, rot_mat))
    ax.imshow(rgb_rot, interpolation='none')

In [ ]:
test_image_warping.TestComputeRotationMatrix.eval(compute_rotation_matrix_fn=compute_rotation_matrix)

# Task3: Transformation estimation in OpenCV

- Re-implement the example of the region of interest (ROI) extraction using transformation estimation that's in the notebook `geometric_transformations.ipynb` **using OpenCV and its functions `findHomography` or `getPerspectiveTransform` and `warpPerspective`**.
- Apply the method to extract the Sudoku grid from the image `data/sudoku-alt3.jpg`.
- Mark the corner points in the image manually.
- The resulting ROI should be a square image with size 288x288.

<figure class="image">
  <img src="../figures/image_warping-expected_transf_est_outputs.png" alt="" style="width: 12.8in;"/>
  <figcaption>Figure 3: Expected outputs.</figcaption>
</figure>

In [ ]:
def rectify_cv(
    img: np.ndarray,
    src_points: np.ndarray,
    dst_size: tuple[int, int],
) -> np.ndarray:
    """
    Rectify the region of interest (ROI) in the input image using OpenCV functions.

    Args:
        img: Input image as a numpy array.
        src_points: 4x2 array of corner points in the input image.
        dst_size: Size (width, height) of the output rectified image.
    Returns:
        roi: The rectified output image as a numpy array.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

    return roi

In [ ]:
rgb = cv.imread('../data/sudoku-alt3.jpg')[..., ::-1]
rgb.shape, rgb.dtype, rgb.min(), rgb.max()

In [ ]:
src_pts = ...

In [ ]:
roi = rectify_cv(rgb, src_pts, (270, 270))
roi.shape, roi.dtype, roi.min(), roi.max()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=plt.figaspect(0.5))
axes[0].imshow(rgb)
axes[0].plot(src_pts[:, 0], src_pts[:, 1], 'o', color=(1, 0, 0));
axes[1].imshow(roi);

In [ ]:
test_image_warping.TestRectifyCv.eval(rectify_cv_fn=rectify_cv, src_pts=src_pts)